# Chapter 6 Lab: The Geometry of Dual Frames

This notebook accompanies **Chapter 6**. It builds the energy-paraboloid
and error-ellipse machinery over the dual-frame parameter plane, then
works through all four lab exercises.

Run the setup cell first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from scipy.linalg import null_space
from scipy.optimize import minimize

np.set_printoptions(precision=4, suppress=True)
%matplotlib inline

def frame_operator(F):
    return F @ F.T

def canonical_dual(F):
    return np.linalg.solve(F @ F.T, F)

def dual_family_direction(F):
    return null_space(F)

f1 = np.array([1.0, 0.0])
f2 = np.array([-0.5, np.sqrt(3)/2])
f3 = np.array([-0.5, -np.sqrt(3)/2])
F_tri = np.column_stack([f1, f2, f3])
F_tri_tilde = canonical_dual(F_tri)
k_tri = dual_family_direction(F_tri)[:, 0]

def G_of_w(w, F_tilde, k):
    Z = np.outer(w, k)
    return F_tilde + Z

def energy(w, F_tilde, k):
    G = G_of_w(w, F_tilde, k)
    return np.trace(G @ G.T)

def error_covariance(w, F_tilde, k, sigma=1.0):
    G = G_of_w(w, F_tilde, k)
    return sigma**2 * (G @ G.T)

## Lab Exercise 6.1: Reproducing the Paraboloid

Verify for at least 10 random $w\in\mathbb{R}^2$ that `energy(w)` matches $\tfrac43+\|w\|^2$ to within $10^{-10}$. Find $w$ (other than the origin) with energy exactly $\tfrac{10}{3}$.

In [ ]:
rng = np.random.default_rng(0)
print("=== Verifying energy(w) = 4/3 + ||w||^2 ===")
for _ in range(10):
    w = rng.standard_normal(2) * 2
    actual = None      # TODO: energy(w, F_tri_tilde, k_tri)
    predicted = None   # TODO: 4/3 + np.dot(w, w)
    print(f"w={w.round(3)}: actual={actual:.8f}, predicted={predicted:.8f}, "
          f"match={np.isclose(actual, predicted, atol=1e-10)}")

target_energy = 10/3
needed_norm_sq = target_energy - 4/3
w_target = np.array([np.sqrt(needed_norm_sq), 0.0])
actual_energy = None  # TODO: energy(w_target, F_tri_tilde, k_tri)
print(f"\nw = {w_target}: energy = {actual_energy:.6f} (target: {target_energy:.6f})")

## Lab Exercise 6.2: The Non-Tight Case

For $\{h_1,h_2,h_3\}$ ($A=1,B=3$): (a) find $k$ spanning $\ker H$. (b) verify the paraboloid. (c) plot the error ellipse at $w=0$. (d) search for a circularizing $w$. (e) guess its exact closed form.

In [ ]:
h1, h2, h3 = np.array([1.0,0.0]), np.array([0.0,1.0]), np.array([1.0,1.0])
F_h = np.column_stack([h1, h2, h3])
F_h_tilde = canonical_dual(F_h)

k_h = dual_family_direction(F_h)[:, 0]
print(f"k (spans ker H) = {k_h}")

canonical_energy_h = np.trace(F_h_tilde @ F_h_tilde.T)
print(f"\n||F_h_tilde||_F^2 = {canonical_energy_h:.6f}")
for _ in range(5):
    w = rng.standard_normal(2)
    actual = energy(w, F_h_tilde, k_h)
    predicted = canonical_energy_h + np.dot(w, w)
    print(f"w={w.round(3)}: actual={actual:.8f}, predicted={predicted:.8f}")

Cov0 = error_covariance(np.array([0.0, 0.0]), F_h_tilde, k_h)
eigvals0 = np.linalg.eigvalsh(Cov0)
print(f"\nError covariance at w=0:\n{Cov0}")
print(f"Eigenvalues: {eigvals0}, ratio: {eigvals0[-1]/eigvals0[0]:.4f} (expect ~3)")

def eigenvalue_gap(w):
    Cov = error_covariance(np.array(w), F_h_tilde, k_h)
    eigs = np.linalg.eigvalsh(Cov)
    return (eigs[-1] - eigs[0])**2

result = None  # TODO: minimize(eigenvalue_gap, [0.5, 0.5])
w_star = result.x
print(f"\nw* = {w_star}")
energy_at_star = energy(w_star, F_h_tilde, k_h)
print(f"Energy at w*: {energy_at_star:.6f}  (vs canonical minimum 4/3={4/3:.6f})")
print(f"Extra energy cost: {energy_at_star - 4/3:.6f}")

Cov_star = error_covariance(w_star, F_h_tilde, k_h)
print(f"Covariance at w*:\n{Cov_star}  (should be close to a multiple of I)")

*Part (e): your guess for the exact closed form of w*, and your hand verification that G(w*)G(w*)^T = I:* (replace this text)

## Lab Exercise 6.3: A Larger Parameter Space

Using the square, find a 2-D basis $\{k_1,k_2\}$ of $\ker F\subset\mathbb{R}^4$, parametrize $Z=w_1e_1k_1^T+w_2e_1k_2^T$, and plot the resulting energy function.

In [ ]:
angles4 = np.array([0, np.pi/2, np.pi, 3*np.pi/2])
F_sq = np.vstack([np.cos(angles4), np.sin(angles4)])
F_sq_tilde = canonical_dual(F_sq)

K = None  # TODO: dual_family_direction(F_sq)
k1, k2 = K[:, 0], K[:, 1]
print(f"k1 . k2 = {np.dot(k1, k2):.2e}  (should be ~0, i.e. orthogonal)")

def restricted_energy(w1, w2):
    Z = np.zeros((2, 4))
    Z[0, :] = w1 * k1 + w2 * k2
    G = F_sq_tilde + Z
    return np.trace(G @ G.T)

w1_grid, w2_grid = np.meshgrid(np.linspace(-2, 2, 100), np.linspace(-2, 2, 100))
E_grid = np.zeros_like(w1_grid)
for i in range(w1_grid.shape[0]):
    for j in range(w1_grid.shape[1]):
        E_grid[i, j] = None  # TODO: restricted_energy(w1_grid[i,j], w2_grid[i,j])

plt.figure(figsize=(6,5))
cs = plt.contour(w1_grid, w2_grid, E_grid, levels=12, cmap='viridis')
plt.clabel(cs, inline=True, fontsize=8)
plt.xlabel('w1'); plt.ylabel('w2')
plt.title('Restricted energy function over the (w1,w2) slice')
plt.gca().set_aspect('equal')
plt.show()

*Your answer: is the restricted energy function still an exact paraboloid, and why (in terms of k1, k2's orthogonality)?* (replace this text)

## Lab Exercise 6.4 (Optional, Challenge): Animating the Family

Animate $w$ sweeping a circle of radius $1.5$; show the corresponding error ellipse. Confirm the energy stays exactly constant.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

radius = 1.5
n_frames = 60
ts = np.linspace(0, 2*np.pi, n_frames)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
energies_over_sweep = []

def animate(frame_idx):
    ax1.clear(); ax2.clear()
    t = ts[frame_idx]
    w_t = radius * np.array([np.cos(t), np.sin(t)])

    e = energy(w_t, F_tri_tilde, k_tri)
    energies_over_sweep.append(e)

    ax1.plot(w_t[0], w_t[1], 'o', color='#534AB7', markersize=10)
    circle = plt.Circle((0,0), radius, fill=False, linestyle='--', color='gray')
    ax1.add_patch(circle)
    ax1.set_xlim(-2, 2); ax1.set_ylim(-2, 2); ax1.set_aspect('equal')
    ax1.set_title(f'w(t), energy={e:.4f}')

    Cov = error_covariance(w_t, F_tri_tilde, k_tri)
    eigvals, eigvecs = np.linalg.eigh(Cov)
    angle = np.degrees(np.arctan2(eigvecs[1,-1], eigvecs[0,-1]))
    ell = Ellipse((0,0), 2*np.sqrt(eigvals[-1]), 2*np.sqrt(eigvals[0]),
                  angle=angle, facecolor='#1D9E75', alpha=0.3, edgecolor='#0F6E56')
    ax2.add_patch(ell)
    ax2.set_xlim(-3, 3); ax2.set_ylim(-3, 3); ax2.set_aspect('equal')
    ax2.set_title('Error ellipse')

anim = FuncAnimation(fig, animate, frames=n_frames, interval=80)
plt.close(fig)
HTML(anim.to_jshtml())

In [ ]:
print(f"Energy range during sweep: [{min(energies_over_sweep):.6f}, {max(energies_over_sweep):.6f}]")
print(f"Constant (within 1e-8)? {max(energies_over_sweep) - min(energies_over_sweep) < 1e-8}")